# D1-05 Uncertainty and Monte Carlo basics

This notebook introduces a practitioner-facing uncertainty workflow in `brightway` using the uncertainty information already present in the imported BAFU database.

We move from exchange-level uncertainty, to the `stats_arrays` data model, to stochastic LCIA scores and ranking robustness.

## Learning goals

- Find uncertainty information attached to exchanges in a real database.
- Connect Brightway exchange fields to the `stats_arrays` parameter-array format.
- Sample and visualize the uncertainty of a few individual exchanges.
- Run stochastic LCAs with `brightway` and summarize score distributions.
- Discuss how percentile overlap and a statistical test inform ranking robustness.
- **Optional extension:** use a Sobol-style variance decomposition to identify dominant uncertainty drivers in a reduced model.
- **Optional extension:** inject precomputed samples for selected exchanges while leaving the rest of the model stochastic.

## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- `stats_arrays` documentation: https://stats-arrays.readthedocs.io/en/latest/

## 1) Load the imported BAFU database and choose two comparison activities

To stay close to `D1-04`, we compare two passenger-car operation activities already present in the BAFU workbook: one petrol car and one natural-gas car.

The impact category is climate change (`EF v3.1`, `GWP100`).

In [ ]:
import matplotlib.pyplot as plt # for plotting
import numpy as np # for storing numbers in arrays
import pandas as pd # for creating tables
from IPython.display import display # to control display parameters
from scipy import stats # to access statistical methods from SciPy
from stats_arrays import MCRandomNumberGenerator, UncertaintyBase, uncertainty_choices

import bw2calc as bc
import bw2data as bd
import bw_processing as bwp

pd.options.display.float_format = '{:,.3f}'.format

In [ ]:
bd.projects

In [ ]:
bd.projects.set_current('aalborg-rlcia-2026')
database_name = 'bafu'
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

Let's find our gasoline and natural gas cars...

In [ ]:
petrol_car = [
    act for act in bd.Database("bafu")
    if all(
        x in act["name"].lower() for x in ("gasoline", "passenger car", "medium", "2018")
    )
    and act["unit"] == "kilometer"
    and act["location"] == "RER"
    and all(y not in act["name"] for y in ("fleet", "hybrid"))
][0]

ng_car = [
    act for act in bd.Database("bafu")
    if all(
        x in act["name"].lower() for x in ("compressed gas", "passenger car", "medium", "2018")
    )
    and act["unit"] == "kilometer"
    and act["location"] == "RER"
    and all(y not in act["name"] for y in ("fleet", "hybrid", "biomethane"))
][0]

In [ ]:
petrol_car

In [ ]:
ng_car

## 2) Inspect exchange uncertainty and the `stats_arrays` data model

Brightway stores uncertainty directly on exchanges, using fields such as `uncertainty type`, `loc`, `scale`, `minimum`, and `maximum`.

These fields follow the [`stats_arrays`](https://stats-arrays.readthedocs.io/en/latest/) convention.  

| Name                      | ID | loc          | scale       | shape | minimum            | maximum            |
| ------------------------- | -: | ------------ | ----------- | ----- | ------------------ | ------------------ |
| Undefined                 |  0 | static value |             |       |                    |                    |
| No uncertainty            |  1 | static value |             |       |                    |                    |
| Lognormal                 |  2 | mu           | sigma       |       | lower bound (opt.) | upper bound (opt.) |
| Normal                    |  3 | mu           | sigma       |       | lower bound (opt.) | upper bound (opt.) |
| Uniform                   |  4 |              |             |       | minimum            | maximum            |
| Triangular                |  5 | mode         |             |       | minimum            | maximum            |
| Bernoulli                 |  6 | p            |             |       | lower bound (opt.) | upper bound (opt.) |
| Discrete uniform          |  7 |              |             |       | minimum            | maximum            |
| Weibull                   |  8 | offset       | lambda      | k     |                    |                    |
| Gamma                     |  9 | offset       | theta       | k     |                    |                    |
| Beta                      | 10 | alpha        | upper bound | beta  |                    |                    |
| Generalized extreme value | 11 | mu           | sigma       | xi    |                    |                    |
| Student's t               | 12 | median       | scale       | nu    |                    |                    |


## 3) Sample a few exchange distributions directly

Before running full Monte Carlo LCAs, it is useful to look at the uncertainty of a few individual exchanges.

Here we stay at the exchange level and let `stats_arrays` sample three uncertain biosphere exchanges from the petrol car activity.

First, let's list the type of uncertainty distribution `stats_array` is able to represent.

In [ ]:
import stats_arrays

In [ ]:
for distribution in stats_arrays.uncertainty_choices:
    print(distribution.id, distribution.__name__)

In [ ]:
# let's build a dictionary where we store the distribution's id and name
uncertainy_type_names = {
    distribution.id: distribution.__name__ 
    for distribution in stats_arrays.uncertainty_choices
}

# let's select the first three biosphere exchanges of our petrol car
# we sort them by descending order amount-wise
selected_exchanges = sorted(
    petrol_car.biosphere(),
    key=lambda exc: exc["amount"],
    reverse=True,
)[:3]

selected_labels = [
    f"{exc['name']} \n{exc['categories'][-1]}" 
    for exc in selected_exchanges
]

selected_amounts = [exc['amount'] for exc in selected_exchanges]

# let's store their uncertainty distribution parameters
selected_params = UncertaintyBase.from_dicts(
    *(exc.as_dict() for exc in selected_exchanges)
)

In [ ]:
# let's make a dataframe out of those
selected_table = pd.DataFrame.from_records(selected_params)
selected_table.insert(0, 'exchange', selected_labels)
selected_table.insert(1, 'amount', [x for x in selected_amounts])

selected_table['distribution'] = selected_table['uncertainty_type'].map(uncertainy_type_names)

display(selected_table[['exchange', 'amount', 'distribution', 'uncertainty_type', 'loc', 'scale', 'minimum', 'maximum']])

We can randomly sample from those distirbutions and plot that.

In [ ]:
draws = 5000
rng = MCRandomNumberGenerator(selected_params)
samples = np.vstack([rng.next() for _ in range(draws)])

fig, axes = plt.subplots(1, len(selected_labels), figsize=(12, 3.5))
axes = np.atleast_1d(axes)

for idx, ax in enumerate(axes):
    ax.hist(samples[:, idx], bins=40, color='#4C78A8', alpha=0.8)
    ax.axvline(selected_amounts[idx], color='#F58518', linestyle='--', linewidth=2)
    ax.set_title(selected_labels[idx])
    ax.set_xlabel('Sampled amount')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4) Run stochastic LCAs in Brightway

Now we propagate uncertainty through the full inventory and impact assessment calculation.

`bw2calc.LCA` does stochastic LCA when `bc.LCA(..., use_distributions=True)` and repeated calls to `next(lca)`.

We also fix the random seed so everyone in the course sees the same Monte Carlo sequence.

> **Why this code exists**
>
> - **Input:** one functional unit, one method, an iteration count, and a seed.
> - **Purpose:** wrap Brightway's repeated stochastic calculation once so both vehicles are evaluated consistently.
> - **Produces:** two score arrays whose rows represent comparable Monte Carlo iterations.
> - **If it fails:** rerun the activity-selection cells and verify that `bw2calc` can complete one ordinary LCA first.


In [ ]:
def stochastic_scores(fu, method, iterations=300, seed=1234):
    lca = bc.JacobiGMRESLCA(
        fu,
        method=method,
        use_distributions=True,
        seed_override=seed,
    )
    lca.lci()
    lca.lcia()

    scores = [lca.score]
    for _ in range(iterations - 1):
        next(lca)
        scores.append(lca.score)

    return np.array(scores, dtype=float)

In [ ]:
%%time

iterations = 300 # number of iterations
seed = 1234 # our seed number

petrol_scores = stochastic_scores(
    fu={petrol_car: 1},
    method=method,
    iterations=iterations,
    seed=seed
)
gas_scores = stochastic_scores(
    fu={ng_car: 1},
    method=method,
    iterations=iterations,
    seed=seed
)

summary = pd.DataFrame([
    {
        'activity': 'Petrol car',
        'mean': petrol_scores.mean(),
        'std': petrol_scores.std(ddof=1),
        'p05': np.percentile(petrol_scores, 5),
        'p50': np.percentile(petrol_scores, 50),
        'p95': np.percentile(petrol_scores, 95),
    },
    {
        'activity': 'Natural gas car',
        'mean': gas_scores.mean(),
        'std': gas_scores.std(ddof=1),
        'p05': np.percentile(gas_scores, 5),
        'p50': np.percentile(gas_scores, 50),
        'p95': np.percentile(gas_scores, 95),
    },
])

print(f'Random seed used: {seed}')
display(summary)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(petrol_scores, bins=30, alpha=0.6, label='Petrol car')
plt.hist(gas_scores, bins=30, alpha=0.6, label='Natural gas car')
plt.xlabel('Impact score')
plt.ylabel('Frequency')
plt.title('Monte Carlo score distributions')
plt.legend()
plt.tight_layout()
plt.show()

## 5) Compare percentiles and ranking robustness

A first practical check is to compare percentile ranges.

For climate change, lower scores are better. So we also calculate the probability that the natural-gas car scores lower than the petrol car in a random pair of Monte Carlo draws.

In [ ]:

interval_table = pd.DataFrame([
    {
        'activity': 'Petrol car',
        'p05': np.percentile(petrol_scores, 5),
        'p50': np.percentile(petrol_scores, 50),
        'p95': np.percentile(petrol_scores, 95),
    },
    {
        'activity': 'Natural gas car',
        'p05': np.percentile(gas_scores, 5),
        'p50': np.percentile(gas_scores, 50),
        'p95': np.percentile(gas_scores, 95),
    },
])

intervals_overlap = not (
    interval_table.loc[0, 'p95'] < interval_table.loc[1, 'p05']
    or interval_table.loc[1, 'p95'] < interval_table.loc[0, 'p05']
)
probability_gas_better = (gas_scores[:, None] < petrol_scores[None, :]).mean()

display(interval_table)
print('Do the 5th-95th percentile intervals overlap?', intervals_overlap)
print(f'Probability that natural gas scores lower than petrol: {probability_gas_better:.3f}')

## 6) Add a statistical test

One possible next step is a Mann-Whitney U test.

This non-parametric test checks whether the two score distributions are shifted relative to one another, without assuming normality. It should be read together with the overlap and the probability-of-superiority result above, not in isolation.

In [ ]:
mw_result = stats.mannwhitneyu(
    petrol_scores,
    gas_scores,
    alternative="two-sided",
)

n_petrol = len(petrol_scores)
n_gas = len(gas_scores)

# Rank-biserial correlation:
# positive values indicate higher petrol scores
rank_biserial = (
    2 * mw_result.statistic / (n_petrol * n_gas)
) - 1

petrol_median = np.median(petrol_scores)
gas_median = np.median(gas_scores)

print(f"Petrol median: {petrol_median:.4g}")
print(f"Gas median: {gas_median:.4g}")
print(f"Mann–Whitney U statistic: {mw_result.statistic:,.0f}")
print(f"p-value: {mw_result.pvalue:.3e}")
print(f"Rank-biserial correlation: {rank_biserial:.3f}")

if mw_result.pvalue < 0.05:
    if rank_biserial > 0:
        direction = "Petrol scores tend to be higher than gas scores."
    elif rank_biserial < 0:
        direction = "Gas scores tend to be higher than petrol scores."
    else:
        direction = "No directional difference is apparent."

    print("The distributions differ statistically.")
    print(direction)
else:
    print(
        "The test does not provide sufficient evidence "
        "of a difference between the distributions."
    )

The rank-biserial correlation can also be interpreted probabilistically:

In [ ]:
print(f"There's an {int(((rank_biserial + 1)/2)*100)} % chance that the petrol car has a higher GWP than the natural gas car")

Natural gas shows a lower median impact than petrol, and the sampled distributions differ statistically; however, the uncertainty intervals overlap  and the probability of natural gas outperforming petrol is >80%, indicating a strong but not absolute confidence in the ranking.

## 7) Paired stochastic LCA

With possibly shared uncertain parameters, a paired comparison might be more appropriate.


In [ ]:
iterations = 300
seed = 1234

diff_scores = stochastic_scores(
    {petrol_car: 1, ng_car: -1}, # we run the delta of the two systems
    method, 
    iterations=iterations, 
    seed=seed
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(diff_scores, bins=30, alpha=0.6, label='Petrol car vs. Natural gas car')
plt.xlabel('Impact score (<0 Petrol car, is better, >0 Natural gas is better)')
plt.ylabel('Frequency')
plt.title('Monte Carlo score distributions')
plt.legend()
plt.tight_layout()
plt.show()

## 8) Optional extension: precomputed paired samples for selected exchanges

Sometimes we want more control than standard Monte Carlo for a few key exchanges.

Here we precompute structured pseudo-random samples for the **fuel input** of both passenger-car activities, and we derive the corresponding **fossil CO2 emission** from each sampled fuel amount. We then inject these four sample series back into Brightway with `add_persistent_array(...)`.

The key point is that only these four matrix entries are overwritten. All other uncertain exchanges keep their original `brightway` / `stats_arrays` uncertainty distributions because we still run the LCA with `use_distributions=True`.

In [ ]:
def sample_exchange_from_unit_interval(exchange, unit_interval):
    params = UncertaintyBase.from_dicts(exchange.as_dict())
    distribution = uncertainty_choices[int(params['uncertainty_type'][0])]
    return distribution.ppf(params, unit_interval.reshape(1, -1)).reshape(-1)

# we create references to the technosphere exchanges of interest
petrol_fuel_exc = next(exc for exc in petrol_car.technosphere() if 'Fuel supply' in exc.input['name'])
gas_fuel_exc = next(exc for exc in ng_car.technosphere() if 'Fuel supply' in exc.input['name'])

# same for the biosphere exchanges
petrol_co2_exc = next(exc for exc in petrol_car.biosphere() if exc.input['name'] == 'Carbon dioxide, fossil')
gas_co2_exc = next(exc for exc in ng_car.biosphere() if exc.input['name'] == 'Carbon dioxide, fossil')

In [ ]:
paired_iterations = 300
paired_seed = 2026
paired_u = (np.arange(paired_iterations) + 0.5) / paired_iterations
rng = np.random.default_rng(paired_seed)
rng.shuffle(paired_u)

petrol_fuel_samples = sample_exchange_from_unit_interval(petrol_fuel_exc, paired_u)
gas_fuel_samples = sample_exchange_from_unit_interval(gas_fuel_exc, paired_u)

petrol_co2_factor = petrol_co2_exc.as_dict()['amount'] / petrol_fuel_exc.as_dict()['amount']
gas_co2_factor = gas_co2_exc.as_dict()['amount'] / gas_fuel_exc.as_dict()['amount']

petrol_co2_samples = petrol_co2_factor * petrol_fuel_samples
gas_co2_samples = gas_co2_factor * gas_fuel_samples

paired_preview = pd.DataFrame({
    'petrol fuel sample': [x for x in petrol_fuel_samples[:5]],
    'petrol CO2 sample': [x for x in petrol_co2_samples[:5]],
    'natural gas sample': [x for x in gas_fuel_samples[:5]],
    'natural gas CO2 sample': [x for x in gas_co2_samples[:5]],
})

print(f'Paired sample seed: {paired_seed}, iterations: {paired_iterations}')
display(paired_preview)

In [ ]:
# we create a data package for bw2calc to consume
# we use bw_processing for this
dp_paired = bwp.create_datapackage(sequential=True)

# we add peristent arrays without our pre-computed values we want to iterate over
dp_paired.add_persistent_array(
    matrix='technosphere_matrix',
    indices_array=np.array([(petrol_fuel_exc.input.id, petrol_car.id)], dtype=bwp.INDICES_DTYPE),
    data_array=petrol_fuel_samples.reshape((1, -1)),
    flip_array=np.array([True]),
    name='petrol-fuel-override',
)
dp_paired.add_persistent_array(
    matrix='biosphere_matrix',
    indices_array=np.array([(petrol_co2_exc.input.id, petrol_car.id)], dtype=bwp.INDICES_DTYPE),
    data_array=petrol_co2_samples.reshape((1, -1)),
    name='petrol-co2-override',
)
dp_paired.add_persistent_array(
    matrix='technosphere_matrix',
    indices_array=np.array([(gas_fuel_exc.input.id, ng_car.id)], dtype=bwp.INDICES_DTYPE),
    data_array=gas_fuel_samples.reshape((1, -1)),
    flip_array=np.array([True]),
    name='gas-fuel-override',
)
dp_paired.add_persistent_array(
    matrix='biosphere_matrix',
    indices_array=np.array([(gas_co2_exc.input.id, ng_car.id)], dtype=bwp.INDICES_DTYPE),
    data_array=gas_co2_samples.reshape((1, -1)),
    name='gas-co2-override',
)

petrol_demand, petrol_data_objs, petrol_remapping = bd.prepare_lca_inputs(demand={petrol_car: 1}, method=method)
gas_demand, gas_data_objs, gas_remapping = bd.prepare_lca_inputs(demand={ng_car: 1}, method=method)

In [ ]:
# we create our LCA objects
# and we give them our specially prepared pre-sampled 
# values for the fuel input + CO2 emissions
petrol_controlled_lca = bc.LCA(
    petrol_demand,
    data_objs=petrol_data_objs + [dp_paired],
    remapping_dicts=petrol_remapping,
    use_distributions=True,
    use_arrays=True,
)
gas_controlled_lca = bc.LCA(
    gas_demand,
    data_objs=gas_data_objs + [dp_paired],
    remapping_dicts=gas_remapping,
    use_distributions=True,
    use_arrays=True,
)
petrol_controlled_lca.lci()
petrol_controlled_lca.lcia()
gas_controlled_lca.lci()
gas_controlled_lca.lcia()

In [ ]:
# we build a nice table to keep track of the exchange values used
petrol_fuel_row = petrol_controlled_lca.dicts.product[petrol_fuel_exc.input.id]
petrol_activity_col = petrol_controlled_lca.dicts.activity[petrol_car.id]
petrol_co2_row = petrol_controlled_lca.dicts.biosphere[petrol_co2_exc.input.id]

gas_fuel_row = gas_controlled_lca.dicts.product[gas_fuel_exc.input.id]
gas_activity_col = gas_controlled_lca.dicts.activity[ng_car.id]
gas_co2_row = gas_controlled_lca.dicts.biosphere[gas_co2_exc.input.id]

paired_records = []
for step in range(paired_iterations):
    paired_records.append({
        'iteration': step,
        'petrol fuel': -float(petrol_controlled_lca.technosphere_matrix[petrol_fuel_row, petrol_activity_col]),
        'petrol fossil CO2': float(petrol_controlled_lca.biosphere_matrix[petrol_co2_row, petrol_activity_col]),
        'petrol score': float(petrol_controlled_lca.score),
        'natural gas fuel': -float(gas_controlled_lca.technosphere_matrix[gas_fuel_row, gas_activity_col]),
        'natural gas fossil CO2': float(gas_controlled_lca.biosphere_matrix[gas_co2_row, gas_activity_col]),
        'natural gas score': float(gas_controlled_lca.score),
    })
    if step < paired_iterations - 1:
        next(petrol_controlled_lca)
        next(gas_controlled_lca)

paired_results = pd.DataFrame(paired_records)

petrol_pairing_ok = np.allclose(
    paired_results['petrol fossil CO2'] / paired_results['petrol fuel'],
    petrol_co2_factor,
)
gas_pairing_ok = np.allclose(
    paired_results['natural gas fossil CO2'] / paired_results['natural gas fuel'],
    gas_co2_factor,
)

print('Did the petrol fuel / CO2 pairing stay fixed?', petrol_pairing_ok)
print('Did the natural-gas fuel / CO2 pairing stay fixed?', gas_pairing_ok)
print('This means only the selected fuel and CO2 exchanges were overwritten; the other uncertain exchanges still use their original distributions.')

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(paired_results["petrol score"], bins=30, alpha=0.6, label='Petrol car')
plt.hist(paired_results["natural gas score"], bins=30, alpha=0.6, label='Natural gas car')
plt.xlabel('Impact score')
plt.ylabel('Frequency')
plt.title('Monte Carlo score distributions')
plt.legend()
plt.tight_layout()
plt.show()

We do not necessarily have a better answer than before, but we now know that the carbon balance in both transport activities is respected.

## Recap

After this notebook, you should now be able to:

- connect Brightway uncertainty fields to the `stats_arrays` parameter-array format
- inspect and sample uncertainty for individual exchanges
- run stochastic LCAs in `brightway`
- summarize score distributions with percentiles and histograms
- explain why overlap, probability of superiority, and a statistical test should be read together
- identify the dominant uncertainty drivers in a reduced Sobol-style analysis
- inject precomputed samples for selected exchanges while preserving the uncertainty of the other exchanges